## 模型文本输出与结构化输出

### 1. 导入依赖并加载 `.env`

In [14]:
import os
from dotenv import load_dotenv
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

load_dotenv()

True

### 2. 定义 DeepSeek 模型客户端

In [15]:
deepseek_model = ChatOpenAI(
    model="deepseek-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking":{"type":"disabled"}},
    timeout=15
)

### 3. 定义 Qwen 模型客户端

In [16]:
qwen_model = ChatOpenAI(
    model="qwen3.8-max-0902",
    # model="qwen-plus",
    api_key=os.getenv("QW_API_KEY"),
    base_url=os.getenv("QW_BASE_URL"),
    timeout=15
)

---
## 4. 普通文本输出
将 DeepSeek 与 Qwen 的普通同步文本输出

### 4.1 定义文本测试 Prompt

In [ ]:
text_prompt = "请用三点简洁说明：为什么人要活着？"

### 1.2 DeepSeek 文本输出（同步 invoke）

In [ ]:
ds_resp = deepseek_model.invoke([HumanMessage(content=text_prompt)])

print(f"模型: {ds_resp.response_metadata.get('model_name')}")
print(ds_resp.content)

### 1.3 Qwen 文本输出（同步 invoke）

In [ ]:
qw_resp = qwen_model.invoke([HumanMessage(content=text_prompt)])

print(f"模型: {qw_resp.response_metadata.get('model_name')}")
print(qw_resp.content)

---
## 2. 结构化输出
先定义统一的 Pydantic Schema，再分别测试 **模型直接输出（`with_structured_output`）** 与 **Agent 输出（`create_agent` + `response_format`）**。

### 2.1 定义结构化输出 Schema 与测试输入

In [17]:
class User(BaseModel):
    """用户信息结构化模型"""
    name: str = Field(description="用户姓名")
    age: int = Field(description="用户年龄")
    skills: list[str] = Field(description="用户掌握的技能列表")


structured_prompt = "请提取用户信息：我叫张三，今年28岁，擅长 Python、LangChain 和 SQL。"


### 2.2 DeepSeek 结构化输出（`with_structured_output`）
- 1. langchain-openai 的 with_structured_output() 默认使用 OpenAI 的 json_schema 模式，DeepSeek 官方接口目前不支持
- 2. deepseek-flash 默认开启思考模式，而 DeepSeek 在思考模式下禁止强制指定 tool_choice 而method="function_calling" 底层会自动传入 tool_choice={"type": "function", "function": {"name": "User"}}，直接触发 400。

In [ ]:
import json

ds_structured_llm = deepseek_model.with_structured_output(User,method="function_calling")

ds_result: User = ds_structured_llm.invoke([HumanMessage(content=structured_prompt)])
# print(f"[DeepSeek]返回类型: {type(ds_result).__name__}")
# print(ds_result.model_dump_json(indent=2, ensure_ascii=False))


# 获取模型额外参数信息
# tools = ds_structured_llm.first.kwargs['tools']
# print(json.dumps(tools, indent=2, ensure_ascii=False))

LangChain 在底层会调用 convert_to_openai_tool(User)，自动自动映射成 OpenAI 要求的 tools 结构：

In [ ]:
import json
from langchain_core.utils.function_calling import convert_to_openai_tool

tool_schema = convert_to_openai_tool(User)
print(json.dumps(tool_schema, indent=2, ensure_ascii=False))

### 2.3 Qwen 结构化输出（`with_structured_output`）
- qwen4-max、Qwen3.7-Plus 系列、Qwen3.8-Flash 系列、Qwen3.7-Flash 系列、Qwen3.7-Max 系列、Qwen3.8-Max 系列，以及 DeepSeek：deepseek-v4.1-flash、deepseek-v4.1-flash-1017、deepseek-v4-pro支持 json schema

In [ ]:
qw_structured_llm = qwen_model.with_structured_output(User)

qw_result: User = qw_structured_llm.invoke([HumanMessage(content=structured_prompt)])

print(f"[Qwen]返回类型: {type(qw_result).__name__}")
print(qw_result.model_dump_json(indent=2, ensure_ascii=False))

### 2.4 DeepSeek Agent 结构化输出（`create_agent` + `response_format`）

In [20]:
ds_agent = create_agent(
    model=deepseek_model,
    tools=[],
    response_format=User
)
ds_agent_res = ds_agent.invoke({"messages": [HumanMessage(content=structured_prompt)]})
structured_data: User = ds_agent_res["structured_response"]
print(f"[DeepSeek Agent]返回类型: {type(structured_data).__name__}")
print(structured_data.model_dump_json(indent=2, ensure_ascii=False))

[DeepSeek Agent]返回类型: User
{
  "name": "张三",
  "age": 28,
  "skills": [
    "Python",
    "LangChain",
    "SQL"
  ]
}


### 2.5 Qwen Agent 结构化输出（`create_agent` + `response_format`）

In [21]:
qw_agent = create_agent(
    model=qwen_model,
    tools=[],
    response_format=User
)
qw_agent_res = qw_agent.invoke({"messages": [HumanMessage(content=structured_prompt)]})

structured_data: User = qw_agent_res["structured_response"]
print(f"[Qwen Agent]返回类型: {type(structured_data).__name__}")
print(structured_data.model_dump_json(indent=2, ensure_ascii=False))

[Qwen Agent]返回类型: User
{
  "name": "张三",
  "age": 28,
  "skills": [
    "Python",
    "LangChain",
    "SQL"
  ]
}
